<a href="https://colab.research.google.com/github/Rontim/GPU-Parallel-Processing-AI/blob/main/gpu_programming/gpu_benchmarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPU Benchmarking Suite
# ======================
## A comprehensive benchmarking toolkit for comparing GPU vs CPU performance

## Setup

In [ ]:
import numpy as np
import cupy as cp
import torch
import time
import platform
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from tabulate import tabulate
from IPython.display import HTML, display

In [ ]:
# Set notebook styling for presentation
display(HTML("""
<style>
.benchmark-title {
    color: #00796B;
    font-size: 24px;
    font-weight: bold;
    padding: 10px 0;
    border-bottom: 2px solid #00796B;
    margin-bottom: 20px;
}
.test-title {
    color: #00695C;
    font-size: 20px;
    font-weight: bold;
    padding: 8px 0;
    margin-top: 15px;
    margin-bottom: 10px;
}
.result-box {
    background-color: #E0F2F1;
    padding: 15px;
    border-radius: 5px;
    margin: 10px 0;
}
</style>
"""))

def print_benchmark(title):
    """Display a formatted benchmark title"""
    display(HTML(f'<div class="benchmark-title">{title}</div>'))

def print_test(title):
    """Display a formatted test title"""
    display(HTML(f'<div class="test-title">{title}</div>'))

def print_result(content):
    """Display a formatted result box"""
    display(HTML(f'<div class="result-box">{content}</div>'))

In [ ]:
# Check GPU availability and initialize
print("🔍 Checking GPU availability...")

gpu_available = cp.cuda.is_available()
print(f"CuPy GPU available: {gpu_available}")

torch_gpu_available = torch.cuda.is_available()
print(f"PyTorch GPU available: {torch_gpu_available}")

if gpu_available:
    gpu_count = cp.cuda.runtime.getDeviceCount()
    print(f"Number of CUDA-enabled GPUs detected: {gpu_count}")
    for i in range(gpu_count):
        print(f"GPU {i}: {cp.cuda.runtime.getDeviceProperties(i)['name'].decode()}")
else:
    print("⚠️ No GPU detected! Benchmarks will run on CPU only.")



## 🔬 Benchmark 1: Matrix Multiplication

In [ ]:
def benchmark_matrix_multiplication(sizes=[1000, 2000, 4000, 6000, 8000]):
    """Benchmark matrix multiplication at different sizes"""
    print_test("Matrix Multiplication Performance Test")
    print("Running matrix multiplication with different sizes...")

    results = []

    for n in sizes:
        print(f"\nTesting size {n}x{n}...")

        # CPU (NumPy)
        A_cpu = np.random.rand(n, n).astype(np.float32)
        B_cpu = np.random.rand(n, n).astype(np.float32)

        # Warm-up
        _ = np.dot(A_cpu[:100, :100], B_cpu[:100, :100])

        # Benchmark
        start_cpu = time.time()
        C_cpu = np.dot(A_cpu, B_cpu)
        cpu_time = time.time() - start_cpu
        print(f"CPU time: {cpu_time:.4f} seconds")

        # GPU (CuPy)
        if gpu_available:
            A_gpu = cp.asarray(A_cpu)
            B_gpu = cp.asarray(B_cpu)

            # Warm-up
            _ = cp.dot(A_gpu[:100, :100], B_gpu[:100, :100])
            cp.cuda.Device(0).synchronize()

            # Benchmark
            start_gpu = time.time()
            C_gpu = cp.dot(A_gpu, B_gpu)
            cp.cuda.Device(0).synchronize()
            gpu_time = time.time() - start_gpu
            print(f"GPU time: {gpu_time:.4f} seconds")

            # Verify results
            C_gpu_cpu = cp.asnumpy(C_gpu)
            max_diff = np.max(np.abs(C_cpu - C_gpu_cpu))
            print(f"Max difference: {max_diff}")

            # Speed-up
            speedup = cpu_time / gpu_time
            print(f"GPU speedup: {speedup:.2f}x")
        else:
            gpu_time = None
            max_diff = None
            speedup = None

        results.append({
            "Matrix Size": f"{n}x{n}",
            "CPU Time (s)": cpu_time,
            "GPU Time (s)": gpu_time,
            "Speedup": speedup,
            "Max Error": max_diff
        })

    # Create DataFrame and display
    results_df = pd.DataFrame(results)
    print("\n📊 Results Summary:")
    print(tabulate(results_df, headers="keys", tablefmt="pretty", showindex=False))

    # Plot results
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sizes_str = [r["Matrix Size"] for r in results]
    plt.plot(sizes_str, [r["CPU Time (s)"] for r in results], 'o-', label='CPU (NumPy)')
    if gpu_available:
        plt.plot(sizes_str, [r["GPU Time (s)"] for r in results], 's-', label='GPU (CuPy)')
    plt.xlabel('Matrix Size')
    plt.ylabel('Time (seconds)')
    plt.title('Matrix Multiplication Performance')
    plt.legend()
    plt.grid(True)

    if gpu_available:
        plt.subplot(1, 2, 2)
        plt.bar(sizes_str, [r["Speedup"] for r in results])
        plt.xlabel('Matrix Size')
        plt.ylabel('Speedup (x times)')
        plt.title('GPU Speedup over CPU')
        plt.grid(True)

    plt.tight_layout()
    plt.show()

    # Generate result summary
    if gpu_available:
        max_speedup = max([r["Speedup"] for r in results])
        avg_speedup = sum([r["Speedup"] for r in results]) / len(results)
        result_text = f"""
        <h3>Matrix Multiplication Benchmark Results</h3>
        <p><strong>Maximum GPU Speedup:</strong> {max_speedup:.2f}x faster than CPU</p>
        <p><strong>Average GPU Speedup:</strong> {avg_speedup:.2f}x faster than CPU</p>
        <p><strong>Maximum Matrix Size:</strong> {sizes[-1]}x{sizes[-1]}</p>
        <p><strong>Numerical Accuracy:</strong> {max([r["Max Error"] for r in results])}</p>
        """
        print_result(result_text)

    return results_df

In [ ]:
matrix_results = benchmark_matrix_multiplication()

## 🔬 Benchmark 2: Element-wise Operations

In [ ]:
def benchmark_elementwise_operations(sizes=[100_000, 1_000_000, 10_000_000, 50_000_000]):
    """Benchmark element-wise operations at different sizes"""
    print_test("Element-wise Operations Performance Test")
    print("Running element-wise operations with different array sizes...")

    results = []

    for n in sizes:
        print(f"\nTesting array size {n}...")

        # CPU (NumPy)
        array_cpu = np.random.rand(n).astype(np.float32)

        # Warm-up
        _ = np.sin(array_cpu[:1000]) + np.exp(array_cpu[:1000]) * np.log(array_cpu[:1000] + 1)

        # Benchmark
        start_cpu = time.time()
        result_cpu = np.sin(array_cpu) + np.exp(array_cpu) * np.log(array_cpu + 1)
        cpu_time = time.time() - start_cpu
        print(f"CPU time: {cpu_time:.4f} seconds")

        # GPU (CuPy)
        if gpu_available:
            array_gpu = cp.asarray(array_cpu)

            # Warm-up
            _ = cp.sin(array_gpu[:1000]) + cp.exp(array_gpu[:1000]) * cp.log(array_gpu[:1000] + 1)
            cp.cuda.Device(0).synchronize()

            # Benchmark
            start_gpu = time.time()
            result_gpu = cp.sin(array_gpu) + cp.exp(array_gpu) * cp.log(array_gpu + 1)
            cp.cuda.Device(0).synchronize()
            gpu_time = time.time() - start_gpu
            print(f"GPU time: {gpu_time:.4f} seconds")

            # Verify results
            result_gpu_cpu = cp.asnumpy(result_gpu)
            max_diff = np.max(np.abs(result_cpu - result_gpu_cpu))
            print(f"Max difference: {max_diff}")

            # Speed-up
            speedup = cpu_time / gpu_time
            print(f"GPU speedup: {speedup:.2f}x")
        else:
            gpu_time = None
            max_diff = None
            speedup = None

        results.append({
            "Array Size": n,
            "Array Size (MB)": n * 4 / (1024**2),  # 4 bytes per float32
            "CPU Time (s)": cpu_time,
            "GPU Time (s)": gpu_time,
            "Speedup": speedup,
            "Max Error": max_diff
        })

    # Create DataFrame and display
    results_df = pd.DataFrame(results)
    print("\n📊 Results Summary:")
    print(tabulate(results_df, headers="keys", tablefmt="pretty", showindex=False))

    # Plot results
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot([r["Array Size (MB)"] for r in results], [r["CPU Time (s)"] for r in results], 'o-', label='CPU (NumPy)')
    if gpu_available:
        plt.plot([r["Array Size (MB)"] for r in results], [r["GPU Time (s)"] for r in results], 's-', label='GPU (CuPy)')
    plt.xlabel('Array Size (MB)')
    plt.ylabel('Time (seconds)')
    plt.title('Element-wise Operations Performance')
    plt.legend()
    plt.grid(True)

    if gpu_available:
        plt.subplot(1, 2, 2)
        plt.bar([str(r["Array Size"]) for r in results], [r["Speedup"] for r in results])
        plt.xlabel('Array Size')
        plt.ylabel('Speedup (x times)')
        plt.title('GPU Speedup over CPU')
        plt.grid(True)

    plt.tight_layout()
    plt.show()

    # Generate result summary
    if gpu_available:
        max_speedup = max([r["Speedup"] for r in results])
        avg_speedup = sum([r["Speedup"] for r in results]) / len(results)
        result_text = f"""
        <h3>Element-wise Operations Benchmark Results</h3>
        <p><strong>Maximum GPU Speedup:</strong> {max_speedup:.2f}x faster than CPU</p>
        <p><strong>Average GPU Speedup:</strong> {avg_speedup:.2f}x faster than CPU</p>
        <p><strong>Maximum Array Size:</strong> {sizes[-1]} elements ({sizes[-1] * 4 / (1024**2):.2f} MB)</p>
        <p><strong>Numerical Accuracy:</strong> {max([r["Max Error"] for r in results])}</p>
        """
        print_result(result_text)

    return results_df

In [ ]:
elementwise_results = benchmark_elementwise_operations()

##  Benchmark 3: Reduction Operations

In [ ]:
def benchmark_reduction_operations(sizes=[100_000, 1_000_000, 10_000_000, 50_000_000]):
    """Benchmark reduction operations at different sizes"""
    print_test("Reduction Operations Performance Test")
    print("Running reduction operations with different array sizes...")

    operations = ["sum", "mean", "std", "min", "max"]
    results = []

    for n in sizes:
        print(f"\nTesting array size {n}...")
        array_cpu = np.random.rand(n).astype(np.float32)

        for op in operations:
            print(f"Testing {op} operation...")

            # CPU (NumPy)
            start_cpu = time.time()
            if op == "sum":
                result_cpu = np.sum(array_cpu)
            elif op == "mean":
                result_cpu = np.mean(array_cpu)
            elif op == "std":
                result_cpu = np.std(array_cpu)
            elif op == "min":
                result_cpu = np.min(array_cpu)
            elif op == "max":
                result_cpu = np.max(array_cpu)
            cpu_time = time.time() - start_cpu

            # GPU (CuPy)
            if gpu_available:
                array_gpu = cp.asarray(array_cpu)

                cp.cuda.Device(0).synchronize()
                start_gpu = time.time()
                if op == "sum":
                    result_gpu = cp.sum(array_gpu)
                elif op == "mean":
                    result_gpu = cp.mean(array_gpu)
                elif op == "std":
                    result_gpu = cp.std(array_gpu)
                elif op == "min":
                    result_gpu = cp.min(array_gpu)
                elif op == "max":
                    result_gpu = cp.max(array_gpu)
                cp.cuda.Device(0).synchronize()
                gpu_time = time.time() - start_gpu

                # Verify results
                result_gpu_cpu = cp.asnumpy(result_gpu)
                diff = abs(result_cpu - result_gpu_cpu)

                # Speed-up
                speedup = cpu_time / gpu_time
            else:
                gpu_time = None
                diff = None
                speedup = None

            results.append({
                "Array Size": n,
                "Operation": op,
                "CPU Time (s)": cpu_time,
                "GPU Time (s)": gpu_time,
                "Speedup": speedup,
                "Difference": diff
            })

    # Create DataFrame and display
    results_df = pd.DataFrame(results)
    print("\n📊 Results Summary:")

    # Get a pivot table for better visualization
    pivot_df = results_df.pivot_table(
        index=["Operation", "Array Size"],
        values=["CPU Time (s)", "GPU Time (s)", "Speedup"],
        aggfunc="mean"
    ).reset_index()

    print(tabulate(pivot_df, headers="keys", tablefmt="pretty", showindex=False))

    # Plot results
    plt.figure(figsize=(15, 10))

    # Plot speedup by operation and size
    if gpu_available:
        plt.subplot(2, 1, 1)
        for op in operations:
            op_results = results_df[results_df["Operation"] == op]
            plt.plot(op_results["Array Size"], op_results["Speedup"], 'o-', label=op)
        plt.xlabel('Array Size')
        plt.ylabel('Speedup (x times)')
        plt.title('GPU Speedup by Reduction Operation')
        plt.legend()
        plt.grid(True)
        plt.xscale('log')

    # Plot time comparison
    plt.subplot(2, 1, 2)
    sns.barplot(data=pivot_df, x="Array Size", y="CPU Time (s)", hue="Operation")
    plt.title('CPU Time by Operation and Array Size')
    plt.xlabel('Array Size')
    plt.ylabel('Time (seconds)')
    plt.yscale('log')
    plt.grid(True, axis='y')

    plt.tight_layout()
    plt.show()

    # Generate result summary
    if gpu_available:
        # Compute average speedup by operation
        op_speedups = {}
        for op in operations:
            op_results = results_df[results_df["Operation"] == op]
            op_speedups[op] = op_results["Speedup"].mean()

        # Find best and worst operations
        best_op = max(op_speedups.items(), key=lambda x: x[1])
        worst_op = min(op_speedups.items(), key=lambda x: x[1])

        result_text = f"""
        <h3>Reduction Operations Benchmark Results</h3>
        <p><strong>Most accelerated operation:</strong> {best_op[0]} ({best_op[1]:.2f}x speedup)</p>
        <p><strong>Least accelerated operation:</strong> {worst_op[0]} ({worst_op[1]:.2f}x speedup)</p>
        <p><strong>Average speedups by operation:</strong></p>
        <ul>
        """

        for op, speedup in op_speedups.items():
            result_text += f"<li>{op}: {speedup:.2f}x</li>"

        result_text += """
        </ul>
        """

        print_result(result_text)

    return results_df

In [ ]:
reduction_results = benchmark_reduction_operations()

## 🔄 Benchmark 4: Memory Transfer

In [ ]:
def benchmark_memory_transfer(sizes=[1_000, 10_000, 100_000, 1_000_000, 10_000_000, 100_000_000]):
    """Benchmark memory transfer between CPU and GPU"""
    if not gpu_available:
        print("⚠️ No GPU available for memory transfer benchmark")
        return None

    print_test("Memory Transfer Performance Test")
    print("Benchmarking memory transfer between CPU and GPU...")

    results = []

    for n in sizes:
        print(f"\nTesting array size {n}...")

        # Create array on CPU
        array_cpu = np.random.rand(n).astype(np.float32)

        # Host to Device transfer
        start = time.time()
        array_gpu = cp.asarray(array_cpu)
        cp.cuda.Device(0).synchronize()
        h2d_time = time.time() - start

        # Device to Host transfer
        start = time.time()
        array_back = cp.asnumpy(array_gpu)
        d2h_time = time.time() - start

        # Calculate bandwidth
        bytes_transferred = n * 4  # 4 bytes per float32
        h2d_bandwidth = bytes_transferred / h2d_time / (1024**3)  # GB/s
        d2h_bandwidth = bytes_transferred / d2h_time / (1024**3)  # GB/s

        results.append({
            "Array Size": n,
            "Data Size (MB)": bytes_transferred / (1024**2),
            "H2D Time (ms)": h2d_time * 1000,
            "D2H Time (ms)": d2h_time * 1000,
            "H2D Bandwidth (GB/s)": h2d_bandwidth,
            "D2H Bandwidth (GB/s)": d2h_bandwidth
        })

    # Create DataFrame and display
    results_df = pd.DataFrame(results)
    print("\n📊 Results Summary:")
    print(tabulate(results_df, headers="keys", tablefmt="pretty", showindex=False))

    # Plot results
    plt.figure(figsize=(12, 10))

    plt.subplot(2, 1, 1)
    plt.plot([r["Data Size (MB)"] for r in results], [r["H2D Time (ms)"] for r in results], 'o-', label='Host to Device')
    plt.plot([r["Data Size (MB)"] for r in results], [r["D2H Time (ms)"] for r in results], 's-', label='Device to Host')
    plt.xlabel('Data Size (MB)')
    plt.ylabel('Transfer Time (ms)')
    plt.title('Memory Transfer Time')
    plt.legend()
    plt.grid(True)
    plt.xscale('log')
    plt.yscale('log')

    plt.subplot(2, 1, 2)
    plt.plot([r["Data Size (MB)"] for r in results], [r["H2D Bandwidth (GB/s)"] for r in results], 'o-', label='Host to Device')
    plt.plot([r["Data Size (MB)"] for r in results], [r["D2H Bandwidth (GB/s)"] for r in results], 's-', label='Device to Host')
    plt.xlabel('Data Size (MB)')
    plt.ylabel('Bandwidth (GB/s)')
    plt.title('Memory Transfer Bandwidth')
    plt.legend()
    plt.grid(True)
    plt.xscale('log')

    plt.tight_layout()
    plt.show()

    # Generate result summary
    max_h2d = max([r["H2D Bandwidth (GB/s)"] for r in results])
    max_d2h = max([r["D2H Bandwidth (GB/s)"] for r in results])
    peak_size_h2d = results[[r["H2D Bandwidth (GB/s)"] for r in results].index(max_h2d)]["Data Size (MB)"]
    peak_size_d2h = results[[r["D2H Bandwidth (GB/s)"] for r in results].index(max_d2h)]["Data Size (MB)"]

    result_text = f"""
    <h3>Memory Transfer Benchmark Results</h3>
    <p><strong>Peak Host to Device Bandwidth:</strong> {max_h2d:.2f} GB/s (at {peak_size_h2d:.2f} MB)</p>
    <p><strong>Peak Device to Host Bandwidth:</strong> {max_d2h:.2f} GB/s (at {peak_size_d2h:.2f} MB)</p>
    <p><strong>Bandwidth Ratio (H2D/D2H):</strong> {max_h2d/max_d2h:.2f}</p>
    """
    print_result(result_text)

    return results_df


In [ ]:
memory_transfer_results = benchmark_memory_transfer()

## 🧩 Benchmark 5: Custom Kernel Performance

In [ ]:
def benchmark_custom_kernel():
    """Benchmark performance of custom CUDA kernels vs NumPy/CuPy operations"""
    if not gpu_available:
        print("⚠️ No GPU available for custom kernel benchmark")
        return None

    print_test("Custom CUDA Kernel Performance Test")
    print("Comparing built-in functions vs custom CUDA kernels...")

    # Define a simple custom kernel: element-wise multiply and add
    multiply_add_kernel = cp.RawKernel(r'''
    extern "C" __global__
    void multiply_add(const float* x, const float* y, float* out, int n, float a) {
        int tid = blockDim.x * blockIdx.x + threadIdx.x;
        if (tid < n) {
            out[tid] = a * x[tid] + y[tid];
        }
    }
    ''', 'multiply_add')

    # Define test sizes
    sizes = [10_000, 100_000, 1_000_000, 10_000_000, 50_000_000]
    results = []

    for n in sizes:
        print(f"\nTesting array size {n}...")

        # Generate test data
        x_cpu = np.random.rand(n).astype(np.float32)
        y_cpu = np.random.rand(n).astype(np.float32)
        a = 2.5

        # CPU (NumPy) implementation
        start = time.time()
        z_cpu = a * x_cpu + y_cpu
        cpu_time = time.time() - start

        # GPU data
        x_gpu = cp.asarray(x_cpu)
        y_gpu = cp.asarray(y_cpu)
        z_gpu = cp.zeros_like(x_gpu)

        # GPU built-in operation
        cp.cuda.Device(0).synchronize()
        start = time.time()
        z_builtin = a * x_gpu + y_gpu
        cp.cuda.Device(0).synchronize()
        builtin_time = time.time() - start

        # GPU custom kernel
        threads_per_block = 256
        blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

        cp.cuda.Device(0).synchronize()
        start = time.time()
        multiply_add_kernel((blocks_per_grid,), (threads_per_block,),
                           (x_gpu, y_gpu, z_gpu, n, np.float32(a)))
        cp.cuda.Device(0).synchronize()
        kernel_time = time.time() - start

        # Verify results
        z_gpu_cpu = cp.asnumpy(z_gpu)
        max_diff = np.max(np.abs(z_cpu - z_gpu_cpu))

        # Speedups
        cpu_to_builtin = cpu_time / builtin_time
        cpu_to_kernel = cpu_time / kernel_time
        builtin_to_kernel = builtin_time / kernel_time

        results.append({
            "Array Size": n,
            "CPU Time (ms)": cpu_time * 1000,
            "GPU Built-in Time (ms)": builtin_time * 1000,
            "GPU Kernel Time (ms)": kernel_time * 1000,
            "CPU/Built-in Speedup": cpu_to_builtin,
            "CPU/Kernel Speedup": cpu_to_kernel,
            "Built-in/Kernel Ratio": builtin_to_kernel,
            "Max Error": max_diff
        })

    # Create DataFrame and display
    results_df = pd.DataFrame(results)
    print("\n📊 Results Summary:")
    print(tabulate(results_df, headers="keys", tablefmt="pretty", showindex=False))

    # Plot results
    plt.figure(figsize=(12, 10))

    plt.subplot(2, 1, 1)
    plt.plot([r["Array Size"] for r in results], [r["CPU Time (ms)"] for r in results], 'o-', label='CPU (NumPy)')
    plt.plot([r["Array Size"] for r in results], [r["GPU Built-in Time (ms)"] for r in results], 's-', label='GPU Built-in')
    plt.plot([r["Array Size"] for r in results], [r["GPU Kernel Time (ms)"] for r in results], '^-', label='Custom Kernel')
    plt.xlabel('Array Size')
    plt.ylabel('Time (ms)')
    plt.title('Execution Time Comparison')
    plt.legend()
    plt.grid(True)
    plt.xscale('log')
    plt.yscale('log')

    plt.subplot(2, 1, 2)
    plt.plot([r["Array Size"] for r in results], [r["CPU/Built-in Speedup"] for r in results], 'o-', label='CPU/Built-in')
    plt.plot([r["Array Size"] for r in results], [r["CPU/Kernel Speedup"] for r in results], 's-', label='CPU/Kernel')
    plt.plot([r["Array Size"] for r in results], [r["Built-in/Kernel Ratio"] for r in results], '^-', label='Built-in/Kernel')
    plt.axhline(y=1.0, color='r', linestyle='--')  # Reference line at y=1
    plt.xlabel('Array Size')
    plt.ylabel('Speedup Ratio')
    plt.title('Speedup Comparison')
    plt.legend()
    plt.grid(True)
    plt.xscale('log')

    plt.tight_layout()
    plt.show()

    # Generate result summary
    avg_builtin_speedup = sum([r["CPU/Built-in Speedup"] for r in results]) / len(results)
    avg_kernel_speedup = sum([r["CPU/Kernel Speedup"] for r in results]) / len(results)
    avg_kernel_vs_builtin = sum([r["Built-in/Kernel Ratio"] for r in results]) / len(results)

    result_text = f"""
    <h3>Custom Kernel Benchmark Results</h3>
    <p><strong>Average CPU to GPU Built-in Speedup:</strong> {avg_builtin_speedup:.2f}x</p>
    <p><strong>Average CPU to Custom Kernel Speedup:</strong> {avg_kernel_speedup:.2f}x</p>
    <p><strong>Average Built-in to Kernel Ratio:</strong> {avg_kernel_vs_builtin:.2f}x
       ({'faster' if avg_kernel_vs_builtin > 1 else 'slower'} than built-in operations)</p>
    <p><strong>Maximum Numerical Error:</strong> {max([r["Max Error"] for r in results])}</p>
    """
    print_result(result_text)

    return results_df


In [ ]:
custom_kernel_results = benchmark_custom_kernel()

## 📈 Comprehensive Performance Analysis

In [ ]:
def generate_comprehensive_analysis():
    """Generate a comprehensive analysis of all benchmarks"""
    print_test("Overall GPU Performance Analysis")

    if not gpu_available:
        print("⚠️ No GPU detected for comprehensive analysis")
        return

    # Collect speedup data from all benchmarks
    speedup_data = []

    # Matrix multiplication
    if 'matrix_results' in globals() and matrix_results is not None:
        for _, row in matrix_results.iterrows():
            if row["Speedup"] is not None:
                speedup_data.append({
                    "Benchmark": "Matrix Multiplication",
                    "Test": f"{row['Matrix Size']}",
                    "Speedup": row["Speedup"]
                })

    # Element-wise operations
    if 'elementwise_results' in globals() and elementwise_results is not None:
        for _, row in elementwise_results.iterrows():
            if row["Speedup"] is not None:
                speedup_data.append({
                    "Benchmark": "Element-wise Operations",
                    "Test": f"{row['Array Size']} elements",
                    "Speedup": row["Speedup"]
                })

    # Reduction operations
    if 'reduction_results' in globals() and reduction_results is not None:
        for _, row in reduction_results.iterrows():
            if row["Speedup"] is not None:
                speedup_data.append({
                    "Benchmark": "Reduction Operations",
                    "Test": f"{row['Operation']} on {row['Array Size']} elements",
                    "Speedup": row["Speedup"]
                })

    # Custom kernel
    if 'custom_kernel_results' in globals() and custom_kernel_results is not None:
        for _, row in custom_kernel_results.iterrows():
            if row["CPU/Kernel Speedup"] is not None:
                speedup_data.append({
                    "Benchmark": "Custom Kernel",
                    "Test": f"{row['Array Size']} elements",
                    "Speedup": row["CPU/Kernel Speedup"]
                })

    # Create DataFrame
    speedup_df = pd.DataFrame(speedup_data)

    if len(speedup_df) == 0:
        print("No valid benchmark data available for analysis")
        return

    # Aggregate by benchmark
    benchmark_summary = speedup_df.groupby("Benchmark")["Speedup"].agg(
        ["mean", "min", "max", "count"]
    ).reset_index()
    benchmark_summary.columns = ["Benchmark", "Average Speedup", "Min Speedup", "Max Speedup", "Tests"]

    print("\n📊 Benchmark Summary:")
    print(tabulate(benchmark_summary, headers="keys", tablefmt="pretty", showindex=False))

    # Plot overall speedup comparison
    plt.figure(figsize=(14, 10))

    plt.subplot(2, 1, 1)
    sns.barplot(data=benchmark_summary, x="Benchmark", y="Average Speedup", palette="viridis")
    plt.ylabel("Average Speedup (x times)")
    plt.title("Average GPU Speedup by Benchmark Type")
    plt.grid(True, axis='y')

    plt.subplot(2, 1, 2)
    sns.boxplot(data=speedup_df, x="Benchmark", y="Speedup", palette="viridis")
    plt.ylabel("Speedup (x times)")
    plt.title("GPU Speedup Distribution by Benchmark Type")
    plt.grid(True, axis='y')

    plt.tight_layout()
    plt.show()

    # Create recommendation based on results
    overall_avg = speedup_df["Speedup"].mean()
    max_benchmark = benchmark_summary.loc[benchmark_summary["Average Speedup"].idxmax()]
    min_benchmark = benchmark_summary.loc[benchmark_summary["Average Speedup"].idxmin()]

    result_text = f"""
    <h3>Comprehensive GPU Performance Analysis</h3>
    <p><strong>Overall Average Speedup:</strong> {overall_avg:.2f}x faster than CPU</p>
    <p><strong>Best Performing Benchmark:</strong> {max_benchmark['Benchmark']} ({max_benchmark['Average Speedup']:.2f}x speedup)</p>
    <p><strong>Worst Performing Benchmark:</strong> {min_benchmark['Benchmark']} ({min_benchmark['Average Speedup']:.2f}x speedup)</p>

    <h4>Recommendations:</h4>
    <ul>
    """

    # Generate recommendations based on performance patterns
    if max_benchmark["Benchmark"] == "Matrix Multiplication":
        result_text += "<li>This GPU excels at large matrix operations. Consider using it for deep learning and linear algebra workloads.</li>"
    elif max_benchmark["Benchmark"] == "Element-wise Operations":
        result_text += "<li>This GPU performs very well for element-wise operations. It's ideal for image processing, simulations, and general array manipulations.</li>"
    elif max_benchmark["Benchmark"] == "Reduction Operations":
        result_text += "<li>This GPU works best for reduction operations like sum, mean, etc. Good for statistical analysis and data processing workloads.</li>"
    elif max_benchmark["Benchmark"] == "Custom Kernel":
        result_text += "<li>Custom kernels show excellent performance. Consider implementing specialized CUDA kernels for your specific workloads.</li>"

    if min_benchmark["Average Speedup"] < 2:
        result_text += f"<li>The {min_benchmark['Benchmark']} benchmark shows minimal speedup. Consider keeping these operations on CPU to avoid transfer overhead.</li>"

    if 'memory_transfer_results' in globals() and memory_transfer_results is not None:
        result_text += "<li>Consider data transfer costs when designing workflows. Minimize CPU-GPU transfers where possible.</li>"

    result_text += """
    <li>For maximum performance, prioritize keeping data on the GPU for as long as possible during computation chains.</li>
    </ul>
    """

    print_result(result_text)

generate_comprehensive_analysis()


## 📋 Benchmarking Summary

In [ ]:
print_test("Final Summary and Recommendations")

print("""
This GPU benchmarking suite provides a comprehensive assessment of GPU performance across different workloads:

1. *Matrix Multiplication*: Tests linear algebra performance critical for deep learning and scientific computing
2. **Element-wise Operations**: Evaluates performance on point-wise calculations common in data preprocessing
3. **Reduction Operations**: Measures efficiency on operations like sum, mean, min, max that require communication
4. **Memory Transfer**: Quantifies the overhead of moving data between CPU and GPU
5. **Custom Kernel**: Compares performance of custom CUDA code against built-in functions

### Key Takeaways

- **GPU Acceleration**: The benchmarks quantify exactly how much speedup to expect for each operation
- **Memory Bottlenecks**: Transfer costs become significant for smaller computations
- **Workload Optimization**: Identify which operations benefit most from GPU acceleration
- **Scaling Properties**: Understand how performance scales with data size

### Next Steps

- Use these benchmarks to guide optimization efforts in your applications
- Consider implementing custom kernels for critical operations
- Design algorithms to minimize CPU-GPU transfers
- Structure data pipelines to maximize GPU residence time
""")

# Show system information
print("\nTest Environment:")
print(f"- GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode() if gpu_available else 'None'}")
print(f"- CuPy: {cp.__version__}")
print(f"- NumPy: {np.__version__}")
print(f"- Python: {platform.python_version()}")
print(f"- Date: {time.strftime('%Y-%m-%d %H:%M:%S')}")